In [0]:
spark.sql('use catalog f1_catalog')
spark.sql('use schema bronze')

Ingesting circuit file

In [0]:
circuits_df=spark.read \
    .option("header", True) \
    .option('inferSchema',True)\
    .csv("abfss://raw@storagef12.dfs.core.windows.net/circuits/circuits.csv")

In [0]:
from pyspark.sql.functions import col

circuits_df=circuits_df \
    .drop("url") \
    .withColumnRenamed("circuitId", "circuit_id") \
    .withColumnRenamed("circuitRef", "circuit_ref") \
    .withColumnRenamed("lat", "latitude") \
    .withColumnRenamed("lng", "longitude") \
    .withColumnRenamed("alt", "altitude")

In [0]:
from pyspark.sql.functions import current_timestamp

circuits_df=circuits_df.withColumn("ingestion_date", current_timestamp())

In [0]:
circuits_df.display()

In [0]:
circuits_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("circuits")

Ingesting races file

In [0]:
races_df=spark.read \
    .option("header", True) \
    .option('inferSchema',True)\
    .csv("abfss://raw@storagef12.dfs.core.windows.net/races/races.csv")

In [0]:
from pyspark.sql.functions import col

races_df=races_df \
    .drop("url") \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("year", "race_year") \
    .withColumnRenamed("circuitId", "circuit_id")

In [0]:
from pyspark.sql.functions import current_timestamp

races_df=races_df.withColumn("ingestion_date", current_timestamp())

In [0]:
races_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("races")

In [0]:
%sql
select * from f1_catalog.bronze.races

### Ingest Constructors

In [0]:
constructors_df = spark.read \
    .json("abfss://raw@storagef12.dfs.core.windows.net/constructors/constructors.json")

In [0]:
constructors_df=constructors_df \
    .drop("url") \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("constructorRef", "constructor_ref")

In [0]:
from pyspark.sql.functions import current_timestamp

constructors_df=constructors_df.withColumn("ingestion_date", current_timestamp())

In [0]:
display(constructors_df)

In [0]:
%sql
drop table f1_catalog.bronze.constructors

In [0]:
constructors_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("constructors")

### ingest drivers

In [0]:
spark.sql('use catalog f1_catalog')
spark.sql('use schema bronze')

In [0]:
drivers_df = spark.read \
    .json("abfss://raw@storagef12.dfs.core.windows.net/drivers/drivers.json")

In [0]:
display(drivers_df)

In [0]:
from pyspark.sql.functions import col

drivers_df = drivers_df.select(
    col("driverId").alias("driver_id"),
    col("code"),
    col("dob"),
    col("driverRef"),
    col("name.forename").alias("forename"),
    col("name.surname").alias("surname"),
    col("nationality")
)

In [0]:
from pyspark.sql.functions import current_timestamp

drivers_df = drivers_df.withColumn("ingestion_date", current_timestamp())

In [0]:
drivers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("drivers")

In [0]:
%sql 
select * from f1_catalog.bronze.drivers;

### Ingestion results

In [0]:
results_df = spark.read \
    .json("abfss://raw@storagef12.dfs.core.windows.net/results/results.json")

In [0]:
from pyspark.sql.functions import current_timestamp

results_df = results_df.withColumn("ingestion_date", current_timestamp())

In [0]:
results_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("results")

In [0]:
%sql
select * from f1_catalog.bronze.results

### Ingestion Pitstop

In [0]:
pit_stops_df = spark.read \
    .option("multiline", True) \
    .json("abfss://raw@storagef12.dfs.core.windows.net/pitstops/pit_stops.json")

In [0]:
from pyspark.sql.functions import current_timestamp

pit_stops_df = pit_stops_df.withColumn("ingestion_date", current_timestamp())

In [0]:
pit_stops_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("pit_stops")

### Ingest lap time

In [0]:
spark.sql("DROP TABLE f1_catalog.bronze.lap_times")

In [0]:
lap_times_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("abfss://raw@storagef12.dfs.core.windows.net/laptimes")

lap_times_df = lap_times_df.toDF(
    "race_id",
    "driver_id",
    "lap",
    "position",
    "time",
    "milliseconds"
)

In [0]:
from pyspark.sql.functions import current_timestamp

lap_times_df = lap_times_df.withColumn("ingestion_date", current_timestamp())

In [0]:
lap_times_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.bronze.lap_times")

In [0]:
spark.read.table("f1_catalog.bronze.lap_times").columns

### Ingest qualifying

In [0]:
qualifying_df = spark.read \
    .option("multiline", True) \
    .json("abfss://raw@storagef12.dfs.core.windows.net/qualifying")

In [0]:
from pyspark.sql.functions import current_timestamp

qualifying_df = qualifying_df.withColumn("ingestion_date", current_timestamp())

In [0]:
qualifying_df.display()

In [0]:
qualifying_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("qualifying")